In [1]:
from pathlib import Path
import pandas as pd

from adaptive_reward_checkpoint_fresh import (
    build_adaptive_reward_snapshot,
    run_adaptive_reward_from_snapshot,
)
from adaptive_trade_extensions import RollingThresholdConfig, make_threshold_grid

snapshot_path = Path("checkpoints") / "SPY_adaptive_reward_fresh_start_30days.joblib"

common_kwargs = dict(
    daily_csv_path="DataAPI/data/SPY_DAY.csv",
    k5m_csv_path="DataAPI/data/SPY_5M.csv",
    code="SPY",
    daily_chan_start="2008-01-01",
    accumulation_start="2010-01-01",
    N_confirm=5,
    min_labeled_days_to_train=200,
    retrain_every_new_labels=25,
    dp_lookback=5,
    lookahead_days_5m=2.0,
    retrain_every_days_5m=5,
    min_samples_total_5m=300,
    threshold_window_days=2.0,
    threshold_ret_grid=None,
    threshold_min_open_signals=10,
    initial_capital=100000.0,
    fee_pct=0.0,
    daily_chan_max_klines=500,
    five_chan_max_klines=500,
    macro_files={
        "vix_": "VIX.csv",
    },
    static_buy_level=0.20,
    static_sell_level=0.30,
    daily_threshold_config=RollingThresholdConfig(
        lookback_days=30,
        buy_grid=make_threshold_grid(0.05, 0.35, 0.005),
        sell_grid=make_threshold_grid(0.15, 0.60, 0.005),
        min_gap=0.02,
        min_obs=60,
        switch_penalty=0.0,
    ),
    verbose=True,
)


In [2]:
snapshot_res = build_adaptive_reward_snapshot(
    snapshot_path=str(snapshot_path),
    snapshot_end_time="2023-12-31",
    output_dir="output_adaptive_reward_snapshot_build_SPY_30days",
    **common_kwargs,
)

print(snapshot_res["snapshot_path"])
display(snapshot_res["daily_reward_df"].tail())
display(snapshot_res["daily_log_df"].tail())


[TRAIN][DAILY-PROB] n=200 pos=43 (21.50%)
[TRAIN][DAILY-PROB] n=225 pos=49 (21.78%)
[TRAIN][DAILY-PROB] n=250 pos=57 (22.80%)
[TRAIN][DAILY-PROB] n=275 pos=66 (24.00%)
[TRAIN][DAILY-PROB] n=300 pos=74 (24.67%)
[TRAIN][DAILY-PROB] n=325 pos=78 (24.00%)
[TRAIN][DAILY-PROB] n=350 pos=85 (24.29%)
[TRAIN][DAILY-PROB] n=375 pos=94 (25.07%)
[TRAIN][DAILY-PROB] n=400 pos=101 (25.25%)
[TRAIN][DAILY-PROB] n=425 pos=104 (24.47%)
[TRAIN][DAILY-PROB] n=450 pos=112 (24.89%)
[TRAIN][DAILY-PROB] n=475 pos=112 (23.58%)
[TRAIN][DAILY-PROB] n=500 pos=113 (22.60%)
[TRAIN][DAILY-PROB] n=525 pos=124 (23.62%)
[TRAIN][DAILY-PROB] n=550 pos=131 (23.82%)
[TRAIN][DAILY-PROB] n=575 pos=143 (24.87%)
[TRAIN][DAILY-PROB] n=600 pos=150 (25.00%)
[TRAIN][DAILY-PROB] n=625 pos=150 (24.00%)
[TRAIN][DAILY-PROB] n=650 pos=158 (24.31%)
[TRAIN][DAILY-PROB] n=675 pos=160 (23.70%)
[TRAIN][DAILY-PROB] n=700 pos=161 (23.00%)
[TRAIN][DAILY-PROB] n=725 pos=168 (23.17%)
[TRAIN][DAILY-PROB] n=750 pos=173 (23.07%)
[TRAIN][DAILY-PROB]

,date,p_day,buy_level,sell_level,chosen_action,reward_force_buy,reward_free,reward_force_sell,chosen_reward,best_action_ex_post,oracle_equity,close,buy_th_5m,sell_th_5m
3517,2023-12-22,0.150994,0.2,0.3,FORCE_BUY,-0.003475,-0.002643,0.000000,-0.003475,FORCE_SELL,2.550681e+09,473.18,1.95,0.75
3518,2023-12-26,0.136775,0.2,0.3,FORCE_BUY,0.002487,0.002162,0.000190,0.002487,FORCE_BUY,2.557025e+09,475.59,1.55,-0.50
3519,2023-12-27,0.114490,0.2,0.3,FORCE_BUY,0.002649,0.004244,0.000105,0.002649,FREE,2.567878e+09,476.90,1.55,-0.50
3520,2023-12-28,0.139578,0.2,0.3,FORCE_BUY,0.000000,0.000189,-0.000210,0.000000,FREE,2.568363e+09,477.05,-0.50,1.40
3521,2023-12-29,0.154252,0.2,0.3,FORCE_BUY,-0.004881,-0.005516,0.000126,-0.004881,FORCE_SELL,2.568685e+09,475.07,1.80,1.40


,date,equity,cash,pos,buy_th,sell_th,p_day,daily_action,daily_buy_level,daily_sell_level
3517,2023-12-22,810331.755111,0.0,1,1.55,-0.5,0.150994,FORCE_BUY,0.2,0.3
3518,2023-12-26,814458.936162,0.0,1,1.55,-0.5,0.136775,FORCE_BUY,0.2,0.3
3519,2023-12-27,816702.341629,0.0,1,-0.50,1.4,0.114490,FORCE_BUY,0.2,0.3
3520,2023-12-28,816959.220118,0.0,1,1.80,1.4,0.139578,FORCE_BUY,0.2,0.3
3521,2023-12-29,813568.424068,0.0,1,1.80,1.4,0.154252,FORCE_BUY,0.2,0.3


In [3]:
resume_res = run_adaptive_reward_from_snapshot(
    snapshot_path=str(snapshot_path),
    end_time="2026-12-31",
    sim_start="2024-01-01",
    initial_capital=100000.0,
    fee_pct=0.0,
    output_dir="output_adaptive_reward_resumed_fresh_SPY_30days",
    # Optional. If omitted, the code saves next to the source snapshot as
    # checkpoints/SPY_adaptive_reward_fresh_start__continued.joblib
    save_snapshot_path=None,
    verbose=True,
)

print("continued snapshot:", resume_res["continued_snapshot_path"])

display(resume_res["daily_reward_df"].tail())
display(resume_res["daily_log_df"].tail())
display(resume_res["trades_df"].head())


[TRAIN][DAILY-PROB] n=3969 pos=890 (22.42%)
[TRAIN][DAILY-PROB] n=3994 pos=891 (22.31%)
[TRAIN][DAILY-PROB] n=4019 pos=898 (22.34%)
[TRAIN][DAILY-PROB] n=4044 pos=901 (22.28%)
[TRAIN][DAILY-PROB] n=4069 pos=909 (22.34%)
[TRAIN][DAILY-PROB] n=4094 pos=913 (22.30%)
[TRAIN][DAILY-PROB] n=4119 pos=917 (22.26%)
[TRAIN][DAILY-PROB] n=4144 pos=920 (22.20%)
[TRAIN][DAILY-PROB] n=4169 pos=928 (22.26%)
[TRAIN][DAILY-PROB] n=4194 pos=933 (22.25%)
[TRAIN][DAILY-PROB] n=4219 pos=937 (22.21%)
[TRAIN][DAILY-PROB] n=4244 pos=947 (22.31%)
[TRAIN][DAILY-PROB] n=4269 pos=955 (22.37%)
[TRAIN][DAILY-PROB] n=4294 pos=959 (22.33%)
[TRAIN][DAILY-PROB] n=4319 pos=963 (22.30%)
[TRAIN][DAILY-PROB] n=4344 pos=963 (22.17%)
[TRAIN][DAILY-PROB] n=4369 pos=970 (22.20%)
[TRAIN][DAILY-PROB] n=4394 pos=980 (22.30%)
[TRAIN][DAILY-PROB] n=4419 pos=988 (22.36%)
[TRAIN][DAILY-PROB] n=4444 pos=998 (22.46%)
[TRAIN][DAILY-PROB] n=4469 pos=1006 (22.51%)
[TRAIN][DAILY-PROB] n=4494 pos=1015 (22.59%)
[TRAIN][DAILY-PROB] n=4519 pos

,date,p_day,buy_level,sell_level,chosen_action,reward_force_buy,reward_free,reward_force_sell,chosen_reward,best_action_ex_post,oracle_equity,close,buy_th_5m,sell_th_5m
4094,2026-04-15,0.125567,0.2,0.3,FORCE_BUY,0.008710,-0.000604,-0.000604,0.008710,FORCE_BUY,592023.898955,700.65,-0.50,2.00
4095,2026-04-16,0.119610,0.2,0.3,FORCE_BUY,0.001812,0.001528,-0.000257,0.001812,FORCE_BUY,593096.543725,702.22,-0.50,2.00
4096,2026-04-17,0.138897,0.2,0.3,FORCE_BUY,0.011427,0.001167,0.001167,0.011427,FORCE_BUY,599873.873595,710.75,0.85,2.45
4097,2026-04-20,0.138156,0.2,0.3,FORCE_BUY,0.004147,0.004147,0.004147,0.004147,FORCE_BUY,602361.461980,709.49,0.80,2.35
4098,2026-04-21,0.187561,0.2,0.3,FORCE_BUY,-0.003720,-0.003720,-0.003720,-0.003720,FORCE_BUY,600120.559185,707.00,0.80,2.35


,date,equity,cash,pos,buy_th,sell_th,p_day,daily_action,daily_buy_level,daily_sell_level
572,2026-04-15,210746.488312,0.0,1,-0.50,2.00,0.125567,FORCE_BUY,0.2,0.3
573,2026-04-16,211218.724074,0.0,1,0.85,2.45,0.119610,FORCE_BUY,0.2,0.3
574,2026-04-17,213784.438118,0.0,1,0.80,2.35,0.138897,FORCE_BUY,0.2,0.3
575,2026-04-20,213405.446360,0.0,1,0.80,2.35,0.138156,FORCE_BUY,0.2,0.3
576,2026-04-21,212656.486457,0.0,1,0.80,2.35,0.187561,FORCE_BUY,0.2,0.3


,side,seen_idx,exec_px,qty,fee,reason,ts,pred,th,gate,pnl,entry_px,entry_idx
0,buy,643541,469.280,213.092397,0.0,5m BUY signal,2024-01-04 05:45:00,0.576991,0.45,FREE,NaN,NaN,NaN
1,sell,643961,469.080,213.092397,0.0,5m SELL signal,2024-01-08 09:30:00,7.518479,-0.50,FREE,-42.618479,469.280,643541.0
2,buy,643981,470.275,212.550915,0.0,5m BUY signal,2024-01-08 11:10:00,0.467693,0.35,FREE,NaN,NaN,NaN
3,sell,643991,471.080,212.550915,0.0,5m SELL signal,2024-01-08 12:00:00,8.529417,-0.50,FREE,171.103487,470.275,643981.0
4,buy,644317,473.810,211.326238,0.0,5m BUY signal,2024-01-10 08:00:00,1.297413,-0.50,FREE,NaN,NaN,NaN


In [1]:
from pathlib import Path
import pandas as pd

from adaptive_reward_checkpoint_fresh import (
    build_adaptive_reward_snapshot,
    run_adaptive_reward_from_snapshot,
)
from adaptive_trade_extensions import RollingThresholdConfig, make_threshold_grid

snapshot_path = Path("checkpoints") / "QQQ_adaptive_reward_fresh_start_30days.joblib"

common_kwargs = dict(
    daily_csv_path="DataAPI/data/QQQ_DAY.csv",
    k5m_csv_path="DataAPI/data/QQQ_5M.csv",
    code="QQQ",
    daily_chan_start="2008-01-01",
    accumulation_start="2010-01-01",
    N_confirm=5,
    min_labeled_days_to_train=200,
    retrain_every_new_labels=25,
    dp_lookback=5,
    lookahead_days_5m=2.0,
    retrain_every_days_5m=5,
    min_samples_total_5m=300,
    threshold_window_days=2.0,
    threshold_ret_grid=None,
    threshold_min_open_signals=10,
    initial_capital=100000.0,
    fee_pct=0.0,
    daily_chan_max_klines=500,
    five_chan_max_klines=500,
    macro_files={
        "vix_": "VIX.csv",
    },
    static_buy_level=0.20,
    static_sell_level=0.30,
    daily_threshold_config=RollingThresholdConfig(
        lookback_days=30,
        buy_grid=make_threshold_grid(0.05, 0.35, 0.005),
        sell_grid=make_threshold_grid(0.15, 0.60, 0.005),
        min_gap=0.02,
        min_obs=60,
        switch_penalty=0.0,
    ),
    verbose=True,
)


In [2]:
snapshot_res = build_adaptive_reward_snapshot(
    snapshot_path=str(snapshot_path),
    snapshot_end_time="2023-12-31",
    output_dir="output_adaptive_reward_snapshot_build_QQQ_30days",
    **common_kwargs,
)

print(snapshot_res["snapshot_path"])
display(snapshot_res["daily_reward_df"].tail())
display(snapshot_res["daily_log_df"].tail())


[TRAIN][DAILY-PROB] n=200 pos=33 (16.50%)
[TRAIN][DAILY-PROB] n=225 pos=40 (17.78%)
[TRAIN][DAILY-PROB] n=250 pos=46 (18.40%)
[TRAIN][DAILY-PROB] n=275 pos=50 (18.18%)
[TRAIN][DAILY-PROB] n=300 pos=56 (18.67%)
[TRAIN][DAILY-PROB] n=325 pos=59 (18.15%)
[TRAIN][DAILY-PROB] n=350 pos=73 (20.86%)
[TRAIN][DAILY-PROB] n=375 pos=78 (20.80%)
[TRAIN][DAILY-PROB] n=400 pos=89 (22.25%)
[TRAIN][DAILY-PROB] n=425 pos=101 (23.76%)
[TRAIN][DAILY-PROB] n=450 pos=108 (24.00%)
[TRAIN][DAILY-PROB] n=475 pos=114 (24.00%)
[TRAIN][DAILY-PROB] n=500 pos=119 (23.80%)
[TRAIN][DAILY-PROB] n=525 pos=119 (22.67%)
[TRAIN][DAILY-PROB] n=550 pos=125 (22.73%)
[TRAIN][DAILY-PROB] n=575 pos=130 (22.61%)
[TRAIN][DAILY-PROB] n=600 pos=138 (23.00%)
[TRAIN][DAILY-PROB] n=625 pos=144 (23.04%)
[TRAIN][DAILY-PROB] n=650 pos=145 (22.31%)
[TRAIN][DAILY-PROB] n=675 pos=145 (21.48%)
[TRAIN][DAILY-PROB] n=700 pos=149 (21.29%)
[TRAIN][DAILY-PROB] n=725 pos=151 (20.83%)
[TRAIN][DAILY-PROB] n=750 pos=155 (20.67%)
[TRAIN][DAILY-PROB] 

,date,p_day,buy_level,sell_level,chosen_action,reward_force_buy,reward_free,reward_force_sell,chosen_reward,best_action_ex_post,oracle_equity,close,buy_th_5m,sell_th_5m
3517,2023-12-22,0.164813,0.2,0.3,FORCE_BUY,0.003787,0.003836,0.003049,0.003787,FREE,2.003848e+11,408.20,1.8,1.9
3518,2023-12-26,0.160106,0.2,0.3,FORCE_BUY,0.003149,-0.000098,-0.000098,0.003149,FORCE_BUY,2.010158e+11,410.89,1.9,-0.5
3519,2023-12-27,0.139922,0.2,0.3,FORCE_BUY,0.002019,0.000827,0.000827,0.002019,FORCE_BUY,2.014217e+11,411.90,1.9,-0.5
3520,2023-12-28,0.197456,0.2,0.3,FORCE_BUY,-0.002109,0.001019,0.000412,-0.002109,FREE,2.016269e+11,411.68,1.9,-0.5
3521,2023-12-29,0.204547,0.2,0.3,FREE,-0.006967,-0.004135,0.000170,-0.004135,FORCE_SELL,2.016611e+11,409.06,1.9,-0.5


,date,equity,cash,pos,buy_th,sell_th,p_day,daily_action,daily_buy_level,daily_sell_level
3517,2023-12-22,1.797983e+07,0.000000e+00,1,1.9,-0.5,0.164813,FORCE_BUY,0.2,0.3
3518,2023-12-26,1.809831e+07,0.000000e+00,1,1.9,-0.5,0.160106,FORCE_BUY,0.2,0.3
3519,2023-12-27,1.814280e+07,0.000000e+00,1,1.9,-0.5,0.139922,FORCE_BUY,0.2,0.3
3520,2023-12-28,1.813311e+07,0.000000e+00,1,1.9,-0.5,0.197456,FORCE_BUY,0.2,0.3
3521,2023-12-29,1.806910e+07,1.806910e+07,0,1.9,-0.5,0.204547,FREE,0.2,0.3


In [3]:
resume_res = run_adaptive_reward_from_snapshot(
    snapshot_path=str(snapshot_path),
    end_time="2026-12-31",
    sim_start="2024-01-01",
    initial_capital=100000.0,
    fee_pct=0.0,
    output_dir="output_adaptive_reward_resumed_fresh_QQQ_30days",
    # Optional. If omitted, the code saves next to the source snapshot as
    # checkpoints/QQQ_adaptive_reward_fresh_start__continued.joblib
    save_snapshot_path=None,
    verbose=True,
)

print("continued snapshot:", resume_res["continued_snapshot_path"])

display(resume_res["daily_reward_df"].tail())
display(resume_res["daily_log_df"].tail())
display(resume_res["trades_df"].head())


[TRAIN][DAILY-PROB] n=4002 pos=843 (21.06%)
[TRAIN][DAILY-PROB] n=4027 pos=844 (20.96%)
[TRAIN][DAILY-PROB] n=4052 pos=850 (20.98%)
[TRAIN][DAILY-PROB] n=4077 pos=854 (20.95%)
[TRAIN][DAILY-PROB] n=4102 pos=858 (20.92%)
[TRAIN][DAILY-PROB] n=4127 pos=862 (20.89%)
[TRAIN][DAILY-PROB] n=4152 pos=873 (21.03%)
[TRAIN][DAILY-PROB] n=4177 pos=876 (20.97%)
[TRAIN][DAILY-PROB] n=4202 pos=881 (20.97%)
[TRAIN][DAILY-PROB] n=4227 pos=885 (20.94%)
[TRAIN][DAILY-PROB] n=4252 pos=890 (20.93%)
[TRAIN][DAILY-PROB] n=4277 pos=900 (21.04%)
[TRAIN][DAILY-PROB] n=4302 pos=909 (21.13%)
[TRAIN][DAILY-PROB] n=4327 pos=913 (21.10%)
[TRAIN][DAILY-PROB] n=4352 pos=915 (21.02%)
[TRAIN][DAILY-PROB] n=4377 pos=915 (20.90%)
[TRAIN][DAILY-PROB] n=4402 pos=926 (21.04%)
[TRAIN][DAILY-PROB] n=4427 pos=930 (21.01%)
[TRAIN][DAILY-PROB] n=4452 pos=937 (21.05%)
[TRAIN][DAILY-PROB] n=4477 pos=944 (21.09%)
[TRAIN][DAILY-PROB] n=4502 pos=947 (21.04%)
[TRAIN][DAILY-PROB] n=4527 pos=953 (21.05%)
[TRAIN][DAILY-PROB] n=4552 pos=9

,date,p_day,buy_level,sell_level,chosen_action,reward_force_buy,reward_free,reward_force_sell,chosen_reward,best_action_ex_post,oracle_equity,close,buy_th_5m,sell_th_5m
4094,2026-04-15,0.195655,0.2,0.3,FORCE_BUY,0.010239,0.001994,0.000000,0.010239,FORCE_BUY,1.422341e+06,638.3600,0.85,2.50
4095,2026-04-16,0.180615,0.2,0.3,FORCE_BUY,0.001439,0.004868,0.000094,0.001439,FREE,1.429265e+06,640.2400,0.85,2.50
4096,2026-04-17,0.189984,0.2,0.3,FORCE_BUY,0.013857,0.005735,0.002843,0.013857,FORCE_BUY,1.449070e+06,648.9700,0.85,2.50
4097,2026-04-20,0.150256,0.2,0.3,FORCE_BUY,0.005571,0.004654,0.002343,0.005571,FORCE_BUY,1.457143e+06,647.9701,0.85,2.50
4098,2026-04-21,0.181342,0.2,0.3,FORCE_BUY,-0.001865,0.002293,-0.000231,-0.001865,FREE,1.460485e+06,647.4900,-0.50,2.25


,date,equity,cash,pos,buy_th,sell_th,p_day,daily_action,daily_buy_level,daily_sell_level
572,2026-04-15,295368.607460,0.0,1,0.85,2.50,0.195655,FORCE_BUY,0.2,0.3
573,2026-04-16,296238.481797,0.0,1,0.85,2.50,0.180615,FORCE_BUY,0.2,0.3
574,2026-04-17,300277.845077,0.0,1,0.85,2.50,0.189984,FORCE_BUY,0.2,0.3
575,2026-04-20,299815.192231,0.0,1,-0.50,2.25,0.150256,FORCE_BUY,0.2,0.3
576,2026-04-21,299593.050386,0.0,1,-0.50,2.25,0.181342,FORCE_BUY,0.2,0.3


,side,seen_idx,exec_px,qty,fee,reason,ts,pred,th,gate,pnl,entry_px,entry_idx
0,buy,581988,421.2401,237.394303,0.0,5m BUY signal,2024-01-22 13:25:00,1.132745,0.2,FREE,NaN,NaN,NaN
1,sell,581995,422.2700,237.394303,0.0,5m SELL signal,2024-01-22 14:00:00,1.855192,-0.5,FREE,244.492393,421.2401,581988.0
2,buy,581999,421.5700,237.788487,0.0,5m BUY signal,2024-01-22 14:20:00,1.110677,0.2,FREE,NaN,NaN,NaN
3,sell,582009,422.0600,237.788487,0.0,5m SELL signal,2024-01-22 15:10:00,1.354036,-0.5,FREE,116.516359,421.5700,581999.0
4,buy,582018,421.7200,237.980197,0.0,5m BUY signal,2024-01-22 15:55:00,0.827127,0.2,FREE,NaN,NaN,NaN


In [1]:
from pathlib import Path
import pandas as pd

from adaptive_reward_checkpoint_fresh import (
    build_adaptive_reward_snapshot,
    run_adaptive_reward_from_snapshot,
)
from adaptive_trade_extensions import RollingThresholdConfig, make_threshold_grid

snapshot_path = Path("checkpoints") / "QQQ_adaptive_reward_fresh_start_90days_at_2020.joblib"

common_kwargs = dict(
    daily_csv_path="DataAPI/data/QQQ_DAY.csv",
    k5m_csv_path="DataAPI/data/QQQ_5M.csv",
    code="QQQ",
    daily_chan_start="2008-01-01",
    accumulation_start="2010-01-01",
    N_confirm=5,
    min_labeled_days_to_train=200,
    retrain_every_new_labels=25,
    dp_lookback=5,
    lookahead_days_5m=2.0,
    retrain_every_days_5m=5,
    min_samples_total_5m=300,
    threshold_window_days=2.0,
    threshold_ret_grid=None,
    threshold_min_open_signals=10,
    initial_capital=100000.0,
    fee_pct=0.0,
    daily_chan_max_klines=500,
    five_chan_max_klines=500,
    macro_files={
        "vix_": "VIX.csv",
    },
    static_buy_level=0.20,
    static_sell_level=0.30,
    daily_threshold_config=RollingThresholdConfig(
        lookback_days=252,
        buy_grid=make_threshold_grid(0.05, 0.35, 0.005),
        sell_grid=make_threshold_grid(0.15, 0.60, 0.005),
        min_gap=0.02,
        min_obs=60,
        switch_penalty=0.0,
    ),
    verbose=True,
)


In [5]:
snapshot_res = build_adaptive_reward_snapshot(
    snapshot_path=str(snapshot_path),
    snapshot_end_time="2019-12-31",
    output_dir="output_adaptive_reward_snapshot_build_QQQ_90days_at_2020",
    **common_kwargs,
)

print(snapshot_res["snapshot_path"])
display(snapshot_res["daily_reward_df"].tail())
display(snapshot_res["daily_log_df"].tail())


[TRAIN][DAILY-PROB] n=200 pos=33 (16.50%)
[TRAIN][DAILY-PROB] n=225 pos=40 (17.78%)
[TRAIN][DAILY-PROB] n=250 pos=46 (18.40%)
[TRAIN][DAILY-PROB] n=275 pos=50 (18.18%)
[TRAIN][DAILY-PROB] n=300 pos=56 (18.67%)
[TRAIN][DAILY-PROB] n=325 pos=59 (18.15%)
[TRAIN][DAILY-PROB] n=350 pos=73 (20.86%)
[TRAIN][DAILY-PROB] n=375 pos=78 (20.80%)
[TRAIN][DAILY-PROB] n=400 pos=89 (22.25%)
[TRAIN][DAILY-PROB] n=425 pos=101 (23.76%)
[TRAIN][DAILY-PROB] n=450 pos=108 (24.00%)
[TRAIN][DAILY-PROB] n=475 pos=114 (24.00%)
[TRAIN][DAILY-PROB] n=500 pos=119 (23.80%)
[TRAIN][DAILY-PROB] n=525 pos=119 (22.67%)
[TRAIN][DAILY-PROB] n=550 pos=125 (22.73%)
[TRAIN][DAILY-PROB] n=575 pos=130 (22.61%)
[TRAIN][DAILY-PROB] n=600 pos=138 (23.00%)
[TRAIN][DAILY-PROB] n=625 pos=144 (23.04%)
[TRAIN][DAILY-PROB] n=650 pos=145 (22.31%)
[TRAIN][DAILY-PROB] n=675 pos=145 (21.48%)
[TRAIN][DAILY-PROB] n=700 pos=149 (21.29%)
[TRAIN][DAILY-PROB] n=725 pos=151 (20.83%)
[TRAIN][DAILY-PROB] n=750 pos=155 (20.67%)
[TRAIN][DAILY-PROB] 

,date,p_day,buy_level,sell_level,chosen_action,reward_force_buy,reward_free,reward_force_sell,chosen_reward,best_action_ex_post,oracle_equity,close,buy_th_5m,sell_th_5m
2510,2019-12-23,0.159574,0.18,0.225,FORCE_BUY,0.000851,0.000851,0.000851,0.000851,FORCE_BUY,3.878990e+08,211.81,1.25,2.5
2511,2019-12-24,0.132892,0.18,0.225,FORCE_BUY,0.000142,0.000142,0.000142,0.000142,FORCE_BUY,3.879539e+08,211.95,1.25,2.5
2512,2019-12-26,0.138752,0.18,0.225,FORCE_BUY,0.009995,0.000707,0.000707,0.009995,FORCE_BUY,3.918316e+08,214.22,1.25,2.5
2513,2019-12-27,0.200438,0.18,0.225,FREE,-0.005550,0.000882,0.001446,0.000882,FORCE_SELL,3.923981e+08,213.23,1.25,2.5
2514,2019-12-30,0.241057,0.18,0.225,FORCE_SELL,-0.007248,-0.002776,-0.000094,-0.000094,FORCE_SELL,3.923614e+08,212.31,1.25,2.5


,date,equity,cash,pos,buy_th,sell_th,p_day,daily_action,daily_buy_level,daily_sell_level
2510,2019-12-23,1.849386e+06,0.000000e+00,1,1.25,2.5,0.159574,FORCE_BUY,0.18,0.225
2511,2019-12-24,1.850608e+06,0.000000e+00,1,1.25,2.5,0.132892,FORCE_BUY,0.18,0.225
2512,2019-12-26,1.870429e+06,0.000000e+00,1,1.25,2.5,0.138752,FORCE_BUY,0.18,0.225
2513,2019-12-27,1.873827e+06,0.000000e+00,1,1.25,2.5,0.200438,FREE,0.18,0.225
2514,2019-12-30,1.879188e+06,1.879188e+06,0,1.25,2.5,0.241057,FORCE_SELL,0.18,0.225


In [6]:
resume_res = run_adaptive_reward_from_snapshot(
    snapshot_path=str(snapshot_path),
    end_time="2026-12-31",
    sim_start="2020-01-01",
    initial_capital=100000.0,
    fee_pct=0.0,
    output_dir="output_adaptive_reward_resumed_fresh_QQQ_90days_at_2020",
    # Optional. If omitted, the code saves next to the source snapshot as
    # checkpoints/QQQ_adaptive_reward_fresh_start__continued.joblib
    save_snapshot_path=None,
    verbose=True,
)

print("continued snapshot:", resume_res["continued_snapshot_path"])

display(resume_res["daily_reward_df"].tail())
display(resume_res["daily_log_df"].tail())
display(resume_res["trades_df"].head())


[TRAIN][DAILY-PROB] n=2996 pos=634 (21.16%)
[TRAIN][DAILY-PROB] n=3021 pos=639 (21.15%)
[TRAIN][DAILY-PROB] n=3046 pos=642 (21.08%)
[TRAIN][DAILY-PROB] n=3071 pos=643 (20.94%)
[TRAIN][DAILY-PROB] n=3096 pos=645 (20.83%)
[TRAIN][DAILY-PROB] n=3121 pos=653 (20.92%)
[TRAIN][DAILY-PROB] n=3146 pos=656 (20.85%)
[TRAIN][DAILY-PROB] n=3171 pos=663 (20.91%)
[TRAIN][DAILY-PROB] n=3196 pos=674 (21.09%)
[TRAIN][DAILY-PROB] n=3221 pos=675 (20.96%)
[TRAIN][DAILY-PROB] n=3246 pos=678 (20.89%)
[TRAIN][DAILY-PROB] n=3271 pos=685 (20.94%)
[TRAIN][DAILY-PROB] n=3296 pos=688 (20.87%)
[TRAIN][DAILY-PROB] n=3321 pos=694 (20.90%)
[TRAIN][DAILY-PROB] n=3346 pos=694 (20.74%)
[TRAIN][DAILY-PROB] n=3371 pos=697 (20.68%)
[TRAIN][DAILY-PROB] n=3396 pos=700 (20.61%)
[TRAIN][DAILY-PROB] n=3421 pos=711 (20.78%)
[TRAIN][DAILY-PROB] n=3446 pos=714 (20.72%)
[TRAIN][DAILY-PROB] n=3471 pos=719 (20.71%)
[TRAIN][DAILY-PROB] n=3496 pos=725 (20.74%)
[TRAIN][DAILY-PROB] n=3521 pos=728 (20.68%)
[TRAIN][DAILY-PROB] n=3546 pos=7

,date,p_day,buy_level,sell_level,chosen_action,reward_force_buy,reward_free,reward_force_sell,chosen_reward,best_action_ex_post,oracle_equity,close,buy_th_5m,sell_th_5m
4094,2026-04-15,0.185506,0.235,0.260,FORCE_BUY,0.010239,0.001994,0.000000,0.010239,FORCE_BUY,8.998906e+08,638.3600,1.5,2.45
4095,2026-04-16,0.170933,0.235,0.260,FORCE_BUY,0.001439,0.004868,0.000094,0.001439,FREE,9.042711e+08,640.2400,1.5,2.45
4096,2026-04-17,0.180935,0.235,0.260,FORCE_BUY,0.013857,0.005735,0.002843,0.013857,FORCE_BUY,9.168018e+08,648.9700,1.5,2.45
4097,2026-04-20,0.150999,0.235,0.260,FORCE_BUY,0.005571,0.002343,0.002343,0.005571,FORCE_BUY,9.219097e+08,647.9701,1.5,2.45
4098,2026-04-21,0.182351,0.340,0.495,FORCE_BUY,-0.001865,0.002293,-0.000231,-0.001865,FREE,9.240236e+08,647.4900,-0.5,-0.50


,date,equity,cash,pos,buy_th,sell_th,p_day,daily_action,daily_buy_level,daily_sell_level
1579,2026-04-15,2.903031e+06,0.0,1,1.5,2.45,0.185506,FORCE_BUY,0.235,0.260
1580,2026-04-16,2.911581e+06,0.0,1,1.5,2.45,0.170933,FORCE_BUY,0.235,0.260
1581,2026-04-17,2.951282e+06,0.0,1,1.5,2.45,0.180935,FORCE_BUY,0.235,0.260
1582,2026-04-20,2.946735e+06,0.0,1,-0.5,-0.50,0.150999,FORCE_BUY,0.235,0.260
1583,2026-04-21,2.944551e+06,0.0,1,-0.5,-0.50,0.182351,FORCE_BUY,0.340,0.495


,side,seen_idx,exec_px,qty,fee,reason,ts,pred,th,gate,pnl,entry_px,entry_idx
0,buy,389374,212.820,469.880650,0.0,ADAPTIVE_FORCE_BUY->first acceptable 5m signal,2020-01-07 18:35:00,1.959498,1.80,FORCE_BUY,NaN,NaN,NaN
1,sell,389406,215.240,469.880650,0.0,5m SELL signal,2020-01-08 05:15:00,1.380636,-0.50,FREE,1137.111174,212.82,389374.0
2,buy,389426,215.600,469.096063,0.0,5m BUY signal,2020-01-08 07:00:00,2.113084,1.85,FREE,NaN,NaN,NaN
3,sell,389460,216.172,469.096063,0.0,5m SELL signal,2020-01-08 09:50:00,5.489262,-0.50,FREE,268.322948,215.60,389426.0
4,buy,389470,215.770,469.970033,0.0,5m BUY signal,2020-01-08 10:40:00,2.674149,1.85,FREE,NaN,NaN,NaN


In [2]:
resume_res = run_adaptive_reward_from_snapshot(
    dp_lookback_override=252,
    snapshot_path=str(snapshot_path),
    end_time="2026-12-31",
    sim_start="2020-01-01",
    initial_capital=100000.0,
    fee_pct=0.0,
    output_dir="output_adaptive_reward_resumed_fresh_QQQ_252days_at_2020",
    # Optional. If omitted, the code saves next to the source snapshot as
    # checkpoints/QQQ_adaptive_reward_fresh_start__continued.joblib
    save_snapshot_path=None,
    verbose=True,
)

print("continued snapshot:", resume_res["continued_snapshot_path"])

display(resume_res["daily_reward_df"].tail())
display(resume_res["daily_log_df"].tail())
display(resume_res["trades_df"].head())


[TRAIN][DAILY-PROB] n=2996 pos=634 (21.16%)
[TRAIN][DAILY-PROB] n=3021 pos=639 (21.15%)
[TRAIN][DAILY-PROB] n=3046 pos=642 (21.08%)
[TRAIN][DAILY-PROB] n=3071 pos=643 (20.94%)
[TRAIN][DAILY-PROB] n=3096 pos=645 (20.83%)
[TRAIN][DAILY-PROB] n=3121 pos=653 (20.92%)
[TRAIN][DAILY-PROB] n=3146 pos=656 (20.85%)
[TRAIN][DAILY-PROB] n=3171 pos=663 (20.91%)
[TRAIN][DAILY-PROB] n=3196 pos=674 (21.09%)
[TRAIN][DAILY-PROB] n=3221 pos=675 (20.96%)
[TRAIN][DAILY-PROB] n=3246 pos=678 (20.89%)
[TRAIN][DAILY-PROB] n=3271 pos=685 (20.94%)
[TRAIN][DAILY-PROB] n=3296 pos=688 (20.87%)
[TRAIN][DAILY-PROB] n=3321 pos=694 (20.90%)
[TRAIN][DAILY-PROB] n=3346 pos=694 (20.74%)
[TRAIN][DAILY-PROB] n=3371 pos=697 (20.68%)
[TRAIN][DAILY-PROB] n=3396 pos=700 (20.61%)
[TRAIN][DAILY-PROB] n=3421 pos=711 (20.78%)
[TRAIN][DAILY-PROB] n=3446 pos=714 (20.72%)
[TRAIN][DAILY-PROB] n=3471 pos=719 (20.71%)
[TRAIN][DAILY-PROB] n=3496 pos=725 (20.74%)
[TRAIN][DAILY-PROB] n=3521 pos=728 (20.68%)
[TRAIN][DAILY-PROB] n=3546 pos=7

,date,p_day,buy_level,sell_level,chosen_action,reward_force_buy,reward_free,reward_force_sell,chosen_reward,best_action_ex_post,oracle_equity,close,buy_th_5m,sell_th_5m
4094,2026-04-15,0.206469,0.35,0.42,FORCE_BUY,0.010239,0.001994,0.000000,0.010239,FORCE_BUY,7.973647e+08,638.3600,1.5,2.45
4095,2026-04-16,0.193124,0.35,0.42,FORCE_BUY,0.001439,0.004868,0.000094,0.001439,FREE,8.012462e+08,640.2400,1.5,2.45
4096,2026-04-17,0.202771,0.35,0.42,FORCE_BUY,0.013857,0.005735,0.002843,0.013857,FORCE_BUY,8.123492e+08,648.9700,1.5,2.45
4097,2026-04-20,0.170837,0.35,0.42,FORCE_BUY,0.005571,0.002343,0.002343,0.005571,FORCE_BUY,8.168752e+08,647.9701,1.5,2.45
4098,2026-04-21,0.206152,0.35,0.42,FORCE_BUY,-0.001865,0.002293,-0.000231,-0.001865,FREE,8.187482e+08,647.4900,-0.5,-0.50


,date,equity,cash,pos,buy_th,sell_th,p_day,daily_action,daily_buy_level,daily_sell_level
1579,2026-04-15,2.923359e+06,0.0,1,1.5,2.45,0.206469,FORCE_BUY,0.35,0.42
1580,2026-04-16,2.931968e+06,0.0,1,1.5,2.45,0.193124,FORCE_BUY,0.35,0.42
1581,2026-04-17,2.971947e+06,0.0,1,1.5,2.45,0.202771,FORCE_BUY,0.35,0.42
1582,2026-04-20,2.967368e+06,0.0,1,-0.5,-0.50,0.170837,FORCE_BUY,0.35,0.42
1583,2026-04-21,2.965169e+06,0.0,1,-0.5,-0.50,0.206152,FORCE_BUY,0.35,0.42


,side,seen_idx,exec_px,qty,fee,reason,ts,pred,th,gate,pnl,entry_px,entry_idx
0,buy,389374,212.820,469.880650,0.0,ADAPTIVE_FORCE_BUY->first acceptable 5m signal,2020-01-07 18:35:00,1.959498,1.80,FORCE_BUY,NaN,NaN,NaN
1,sell,389406,215.240,469.880650,0.0,5m SELL signal,2020-01-08 05:15:00,1.380636,-0.50,FREE,1137.111174,212.82,389374.0
2,buy,389426,215.600,469.096063,0.0,5m BUY signal,2020-01-08 07:00:00,2.113084,1.85,FREE,NaN,NaN,NaN
3,sell,389460,216.172,469.096063,0.0,5m SELL signal,2020-01-08 09:50:00,5.489262,-0.50,FREE,268.322948,215.60,389426.0
4,buy,389470,215.770,469.970033,0.0,5m BUY signal,2020-01-08 10:40:00,2.674149,1.85,FREE,NaN,NaN,NaN


In [7]:
from pathlib import Path
import pandas as pd

from adaptive_reward_checkpoint_fresh import (
    build_adaptive_reward_snapshot,
    run_adaptive_reward_from_snapshot,
)
from adaptive_trade_extensions import RollingThresholdConfig, make_threshold_grid

snapshot_path = Path("checkpoints") / "TQQQ_adaptive_reward_fresh_start_90days_at_2020.joblib"

common_kwargs = dict(
    daily_csv_path="DataAPI/data/TQQQ_day.csv",
    k5m_csv_path="DataAPI/data/TQQQ_5M.csv",
    code="TQQQ",
    daily_chan_start="2010-02-11",
    accumulation_start="2012-02-11",
    N_confirm=5,
    min_labeled_days_to_train=200,
    retrain_every_new_labels=25,
    dp_lookback=5,
    lookahead_days_5m=2.0,
    retrain_every_days_5m=5,
    min_samples_total_5m=300,
    threshold_window_days=2.0,
    threshold_ret_grid=None,
    threshold_min_open_signals=10,
    initial_capital=100000.0,
    fee_pct=0.0,
    daily_chan_max_klines=500,
    five_chan_max_klines=500,
    macro_files={
        "vix_": "VIX.csv",
    },
    static_buy_level=0.20,
    static_sell_level=0.30,
    daily_threshold_config=RollingThresholdConfig(
        lookback_days=90,
        buy_grid=make_threshold_grid(0.05, 0.35, 0.005),
        sell_grid=make_threshold_grid(0.15, 0.60, 0.005),
        min_gap=0.02,
        min_obs=60,
        switch_penalty=0.0,
    ),
    verbose=True,
)


In [9]:
snapshot_res = build_adaptive_reward_snapshot(
    snapshot_path=str(snapshot_path),
    snapshot_end_time="2019-12-31",
    output_dir="output_adaptive_reward_snapshot_build_TQQQ_90days_at_2020",
    **common_kwargs,
)

print(snapshot_res["snapshot_path"])
display(snapshot_res["daily_reward_df"].tail())
display(snapshot_res["daily_log_df"].tail())

[TRAIN][DAILY-PROB] n=200 pos=49 (24.50%)
[TRAIN][DAILY-PROB] n=225 pos=52 (23.11%)
[TRAIN][DAILY-PROB] n=250 pos=61 (24.40%)
[TRAIN][DAILY-PROB] n=275 pos=65 (23.64%)
[TRAIN][DAILY-PROB] n=300 pos=76 (25.33%)
[TRAIN][DAILY-PROB] n=325 pos=76 (23.38%)
[TRAIN][DAILY-PROB] n=350 pos=76 (21.71%)
[TRAIN][DAILY-PROB] n=375 pos=79 (21.07%)
[TRAIN][DAILY-PROB] n=400 pos=84 (21.00%)
[TRAIN][DAILY-PROB] n=425 pos=91 (21.41%)
[TRAIN][DAILY-PROB] n=450 pos=96 (21.33%)
[TRAIN][DAILY-PROB] n=475 pos=99 (20.84%)
[TRAIN][DAILY-PROB] n=500 pos=104 (20.80%)
[TRAIN][DAILY-PROB] n=525 pos=114 (21.71%)
[TRAIN][DAILY-PROB] n=550 pos=118 (21.45%)
[TRAIN][DAILY-PROB] n=575 pos=120 (20.87%)
[TRAIN][DAILY-PROB] n=600 pos=128 (21.33%)
[TRAIN][DAILY-PROB] n=625 pos=137 (21.92%)
[TRAIN][DAILY-PROB] n=650 pos=142 (21.85%)
[TRAIN][DAILY-PROB] n=675 pos=150 (22.22%)
[TRAIN][DAILY-PROB] n=700 pos=152 (21.71%)
[TRAIN][DAILY-PROB] n=725 pos=155 (21.38%)
[TRAIN][DAILY-PROB] n=750 pos=160 (21.33%)
[TRAIN][DAILY-PROB] n=7

,date,p_day,buy_level,sell_level,chosen_action,reward_force_buy,reward_free,reward_force_sell,chosen_reward,best_action_ex_post,oracle_equity,close,buy_th_5m,sell_th_5m
1978,2019-12-23,0.149319,0.19,0.235,FORCE_BUY,0.002572,0.004443,0.003161,0.002572,FREE,1.702559e+13,10.7212,-0.5,-0.50
1979,2019-12-24,0.140952,0.19,0.235,FORCE_BUY,0.000121,-0.000571,0.002106,0.000121,FORCE_SELL,1.706144e+13,10.7325,-0.5,-0.50
1980,2019-12-26,0.153933,0.19,0.235,FORCE_BUY,0.029743,0.006162,0.001980,0.029743,FORCE_BUY,1.756891e+13,11.0787,-0.5,1.95
1981,2019-12-27,0.202024,0.19,0.235,FREE,-0.014070,0.000797,-0.000117,0.000797,FREE,1.758291e+13,10.9525,-0.5,1.95
1982,2019-12-30,0.226948,0.19,0.235,FREE,-0.021668,-0.029709,-0.027342,-0.029709,FORCE_BUY,1.720192e+13,10.7775,-0.5,1.95


,date,equity,cash,pos,buy_th,sell_th,p_day,daily_action,daily_buy_level,daily_sell_level
1978,2019-12-23,2.230197e+07,0.0,1,-0.5,-0.50,0.149319,FORCE_BUY,0.19,0.235
1979,2019-12-24,2.232547e+07,0.0,1,-0.5,1.95,0.140952,FORCE_BUY,0.19,0.235
1980,2019-12-26,2.304563e+07,0.0,1,-0.5,1.95,0.153933,FORCE_BUY,0.19,0.235
1981,2019-12-27,2.312665e+07,0.0,1,-0.5,1.95,0.202024,FREE,0.19,0.235
1982,2019-12-30,2.257009e+07,0.0,1,-0.5,1.95,0.226948,FREE,0.19,0.235


In [10]:
resume_res = run_adaptive_reward_from_snapshot(
    dp_lookback_override=252,
    snapshot_path=str(snapshot_path),
    end_time="2026-12-31",
    sim_start="2020-01-01",
    initial_capital=100000.0,
    fee_pct=0.0,
    output_dir="output_adaptive_reward_resumed_fresh_TQQQ_90days_at_2020",
    # Optional. If omitted, the code saves next to the source snapshot as
    # checkpoints/TQQQ_adaptive_reward_fresh_start__continued.joblib
    save_snapshot_path=None,
    verbose=True,
)

print("continued snapshot:", resume_res["continued_snapshot_path"])

display(resume_res["daily_reward_df"].tail())
display(resume_res["daily_log_df"].tail())
display(resume_res["trades_df"].head())


[TRAIN][DAILY-PROB] n=2332 pos=470 (20.15%)
[TRAIN][DAILY-PROB] n=2357 pos=475 (20.15%)
[TRAIN][DAILY-PROB] n=2382 pos=477 (20.03%)
[TRAIN][DAILY-PROB] n=2407 pos=478 (19.86%)
[TRAIN][DAILY-PROB] n=2432 pos=480 (19.74%)
[TRAIN][DAILY-PROB] n=2457 pos=484 (19.70%)
[TRAIN][DAILY-PROB] n=2482 pos=487 (19.62%)
[TRAIN][DAILY-PROB] n=2507 pos=493 (19.66%)
[TRAIN][DAILY-PROB] n=2532 pos=504 (19.91%)
[TRAIN][DAILY-PROB] n=2557 pos=505 (19.75%)
[TRAIN][DAILY-PROB] n=2582 pos=508 (19.67%)
[TRAIN][DAILY-PROB] n=2607 pos=516 (19.79%)
[TRAIN][DAILY-PROB] n=2632 pos=519 (19.72%)
[TRAIN][DAILY-PROB] n=2657 pos=525 (19.76%)
[TRAIN][DAILY-PROB] n=2682 pos=525 (19.57%)
[TRAIN][DAILY-PROB] n=2707 pos=528 (19.50%)
[TRAIN][DAILY-PROB] n=2732 pos=531 (19.44%)
[TRAIN][DAILY-PROB] n=2757 pos=542 (19.66%)
[TRAIN][DAILY-PROB] n=2782 pos=545 (19.59%)
[TRAIN][DAILY-PROB] n=2807 pos=551 (19.63%)
[TRAIN][DAILY-PROB] n=2832 pos=557 (19.67%)
[TRAIN][DAILY-PROB] n=2857 pos=559 (19.57%)
[TRAIN][DAILY-PROB] n=2882 pos=5

,date,p_day,buy_level,sell_level,chosen_action,reward_force_buy,reward_free,reward_force_sell,chosen_reward,best_action_ex_post,oracle_equity,close,buy_th_5m,sell_th_5m
3575,2026-05-04,0.250726,0.235,0.255,FREE,-0.017014,-0.008847,-0.005017,-0.008847,FORCE_SELL,2.711665e+16,64.6608,-0.5,-0.5
3576,2026-05-05,0.201556,0.235,0.255,FORCE_BUY,0.000000,0.000000,0.000000,0.000000,FORCE_BUY,2.711665e+16,69.1000,-0.5,-0.5
3577,2026-05-06,0.259130,0.235,0.255,FORCE_SELL,-0.004608,-0.004608,0.000000,0.000000,FORCE_SELL,2.711665e+16,71.2203,-0.5,-0.5
3578,2026-05-07,0.290113,0.235,0.255,FORCE_SELL,0.010347,0.010347,0.000000,0.000000,FORCE_BUY,2.739724e+16,71.2800,-0.5,-0.5
3579,2026-05-08,0.272238,0.235,0.255,FORCE_SELL,0.054989,0.019494,0.000000,0.000000,FORCE_BUY,2.890378e+16,76.5500,-0.5,-0.5


,date,equity,cash,pos,buy_th,sell_th,p_day,daily_action,daily_buy_level,daily_sell_level
1592,2026-05-04,1.559432e+09,1.559432e+09,0,-0.5,-0.5,0.250726,FREE,0.235,0.255
1593,2026-05-05,1.559432e+09,1.559432e+09,0,-0.5,-0.5,0.201556,FORCE_BUY,0.235,0.255
1594,2026-05-06,1.559432e+09,1.559432e+09,0,-0.5,-0.5,0.259130,FORCE_SELL,0.235,0.255
1595,2026-05-07,1.559432e+09,1.559432e+09,0,-0.5,-0.5,0.290113,FORCE_SELL,0.235,0.255
1596,2026-05-08,1.559432e+09,1.559432e+09,0,-0.5,-0.5,0.272238,FORCE_SELL,0.235,0.255


,side,seen_idx,exec_px,qty,fee,reason,ts,pred,th,gate,pnl,entry_px,entry_idx
0,buy,253880,11.2613,8879.969453,0.0,ADAPTIVE_FORCE_BUY->first acceptable 5m signal,2020-01-07 08:30:00,2.143581,-0.5,FORCE_BUY,NaN,NaN,NaN
1,sell,254029,11.2100,8879.969453,0.0,5m SELL signal,2020-01-08 05:15:00,26.507231,-0.5,FREE,-455.542433,11.2613,253880.0
2,buy,254050,11.2625,8838.575589,0.0,5m BUY signal,2020-01-08 07:00:00,0.846646,-0.5,FREE,NaN,NaN,NaN
3,sell,254065,11.3375,8838.575589,0.0,5m SELL signal,2020-01-08 08:15:00,23.341526,-0.5,FREE,662.893169,11.2625,254050.0
4,buy,254070,11.2875,8877.727640,0.0,5m BUY signal,2020-01-08 08:40:00,2.161747,-0.5,FREE,NaN,NaN,NaN


In [ ]:
from pathlib import Path
import pandas as pd

from adaptive_reward_checkpoint_fresh import (
    build_adaptive_reward_snapshot,
    run_adaptive_reward_from_snapshot,
)
from adaptive_trade_extensions import RollingThresholdConfig, make_threshold_grid

snapshot_path = Path("checkpoints") / "QQQ_adaptive_reward_fresh_start_90days_at_2020.joblib"

common_kwargs = dict(
    daily_csv_path="DataAPI/data/QQQ_DAY.csv",
    k5m_csv_path="DataAPI/data/QQQ_5M.csv",
    code="QQQ",
    daily_chan_start="2008-01-01",
    accumulation_start="2010-01-01",
    N_confirm=5,
    min_labeled_days_to_train=200,
    retrain_every_new_labels=25,
    dp_lookback=5,
    lookahead_days_5m=2.0,
    retrain_every_days_5m=5,
    min_samples_total_5m=300,
    threshold_window_days=2.0,
    threshold_ret_grid=None,
    threshold_min_open_signals=10,
    initial_capital=100000.0,
    fee_pct=0.0,
    daily_chan_max_klines=500,
    five_chan_max_klines=500,
    macro_files={
        "vix_": "VIX.csv",
    },
    static_buy_level=0.20,
    static_sell_level=0.30,
    daily_threshold_config=RollingThresholdConfig(
        lookback_days=252,
        buy_grid=make_threshold_grid(0.05, 0.35, 0.005),
        sell_grid=make_threshold_grid(0.15, 0.60, 0.005),
        min_gap=0.02,
        min_obs=60,
        switch_penalty=0.0,
    ),
    autosave_year_start_checkpoints=True,
    verbose=True,
)

snapshot_res = build_adaptive_reward_snapshot(
    snapshot_path=str(snapshot_path),
    snapshot_end_time="2019-12-31",
    output_dir="output_adaptive_reward_snapshot_build_QQQ_90days_at_2020",
    **common_kwargs,
)

print("base snapshot:", snapshot_res["snapshot_path"])
print("year-start build checkpoints:")
for p in snapshot_res["year_start_checkpoint_paths"]:
    print(" ", p)

display(snapshot_res["daily_reward_df"].tail())
display(snapshot_res["daily_log_df"].tail())

resume_res = run_adaptive_reward_from_snapshot(
    snapshot_path=str(snapshot_path),
    end_time="2026-12-31",
    sim_start="2020-01-01",
    initial_capital=100000.0,
    fee_pct=0.0,
    output_dir="output_adaptive_reward_resumed_fresh_QQQ_90days_at_2020",
    save_snapshot_path=None,
    autosave_year_start_checkpoints=True,
    verbose=True,
)

print("continued snapshot:", resume_res["continued_snapshot_path"])
print("year-start resume checkpoints:")
for p in resume_res["year_start_checkpoint_paths"]:
    print(" ", p)

display(resume_res["daily_reward_df"].tail())
display(resume_res["daily_log_df"].tail())
display(resume_res["trades_df"].head())


[TRAIN][DAILY-PROB] n=200 pos=33 (16.50%)
[TRAIN][DAILY-PROB] n=225 pos=40 (17.78%)
[TRAIN][DAILY-PROB] n=250 pos=46 (18.40%)
[TRAIN][DAILY-PROB] n=275 pos=50 (18.18%)
[TRAIN][DAILY-PROB] n=300 pos=56 (18.67%)
[TRAIN][DAILY-PROB] n=325 pos=59 (18.15%)
[TRAIN][DAILY-PROB] n=350 pos=73 (20.86%)
[TRAIN][DAILY-PROB] n=375 pos=78 (20.80%)
[TRAIN][DAILY-PROB] n=400 pos=89 (22.25%)
[TRAIN][DAILY-PROB] n=425 pos=101 (23.76%)
[TRAIN][DAILY-PROB] n=450 pos=108 (24.00%)
[TRAIN][DAILY-PROB] n=475 pos=114 (24.00%)
[TRAIN][DAILY-PROB] n=500 pos=119 (23.80%)
[TRAIN][DAILY-PROB] n=525 pos=119 (22.67%)
[TRAIN][DAILY-PROB] n=550 pos=125 (22.73%)
[TRAIN][DAILY-PROB] n=575 pos=130 (22.61%)
[TRAIN][DAILY-PROB] n=600 pos=138 (23.00%)
[TRAIN][DAILY-PROB] n=625 pos=144 (23.04%)
[TRAIN][DAILY-PROB] n=650 pos=145 (22.31%)
[TRAIN][DAILY-PROB] n=675 pos=145 (21.48%)
[TRAIN][DAILY-PROB] n=700 pos=149 (21.29%)
[TRAIN][DAILY-PROB] n=725 pos=151 (20.83%)
[TRAIN][DAILY-PROB] n=750 pos=155 (20.67%)
[TRAIN][DAILY-PROB] 

,date,p_day,buy_level,sell_level,chosen_action,reward_force_buy,reward_free,reward_force_sell,chosen_reward,best_action_ex_post,oracle_equity,close,buy_th_5m,sell_th_5m
2510,2019-12-23,0.159604,0.185,0.225,FORCE_BUY,0.000851,0.000851,0.000851,0.000851,FORCE_BUY,3.158442e+08,211.81,1.25,2.5
2511,2019-12-24,0.132911,0.185,0.225,FORCE_BUY,0.000142,0.000142,0.000142,0.000142,FORCE_BUY,3.158889e+08,211.95,1.25,2.5
2512,2019-12-26,0.138769,0.185,0.225,FORCE_BUY,0.009995,0.000707,0.000707,0.009995,FORCE_BUY,3.190463e+08,214.22,1.25,2.5
2513,2019-12-27,0.200483,0.185,0.225,FREE,-0.005550,0.000882,0.001446,0.000882,FORCE_SELL,3.195076e+08,213.23,1.25,2.5
2514,2019-12-30,0.241121,0.185,0.225,FORCE_SELL,-0.007248,-0.002776,-0.000094,-0.000094,FORCE_SELL,3.194777e+08,212.31,1.25,2.5


,date,equity,cash,pos,buy_th,sell_th,p_day,daily_action,daily_buy_level,daily_sell_level
2510,2019-12-23,1.860887e+06,0.000000e+00,1,1.25,2.5,0.159604,FORCE_BUY,0.185,0.225
2511,2019-12-24,1.862117e+06,0.000000e+00,1,1.25,2.5,0.132911,FORCE_BUY,0.185,0.225
2512,2019-12-26,1.882060e+06,0.000000e+00,1,1.25,2.5,0.138769,FORCE_BUY,0.185,0.225
2513,2019-12-27,1.885480e+06,0.000000e+00,1,1.25,2.5,0.200483,FREE,0.185,0.225
2514,2019-12-30,1.890874e+06,1.890874e+06,0,1.25,2.5,0.241121,FORCE_SELL,0.185,0.225


[TRAIN][DAILY-PROB] n=2996 pos=634 (21.16%)
[TRAIN][DAILY-PROB] n=3021 pos=639 (21.15%)
[TRAIN][DAILY-PROB] n=3046 pos=642 (21.08%)
[TRAIN][DAILY-PROB] n=3071 pos=643 (20.94%)
[TRAIN][DAILY-PROB] n=3096 pos=645 (20.83%)
[TRAIN][DAILY-PROB] n=3121 pos=653 (20.92%)
[TRAIN][DAILY-PROB] n=3146 pos=656 (20.85%)
[TRAIN][DAILY-PROB] n=3171 pos=663 (20.91%)
[TRAIN][DAILY-PROB] n=3196 pos=674 (21.09%)
[TRAIN][DAILY-PROB] n=3221 pos=675 (20.96%)
[TRAIN][DAILY-PROB] n=3246 pos=678 (20.89%)
[TRAIN][DAILY-PROB] n=3271 pos=685 (20.94%)
[TRAIN][DAILY-PROB] n=3296 pos=688 (20.87%)
[TRAIN][DAILY-PROB] n=3321 pos=694 (20.90%)
[TRAIN][DAILY-PROB] n=3346 pos=694 (20.74%)
[TRAIN][DAILY-PROB] n=3371 pos=697 (20.68%)
[TRAIN][DAILY-PROB] n=3396 pos=700 (20.61%)
[TRAIN][DAILY-PROB] n=3421 pos=711 (20.78%)
[TRAIN][DAILY-PROB] n=3446 pos=714 (20.72%)
[TRAIN][DAILY-PROB] n=3471 pos=719 (20.71%)
[TRAIN][DAILY-PROB] n=3496 pos=725 (20.74%)
[TRAIN][DAILY-PROB] n=3521 pos=728 (20.68%)
[TRAIN][DAILY-PROB] n=3546 pos=7

In [2]:

from pathlib import Path
import pandas as pd

from adaptive_reward_checkpoint_fresh import (
    build_adaptive_reward_snapshot,
    run_adaptive_reward_from_snapshot,
)
from adaptive_trade_extensions import RollingThresholdConfig, make_threshold_grid

snapshot_path = Path("checkpoints") / "QQQ_adaptive_reward_fresh_start_252days_at_2020.joblib"

common_kwargs = dict(
    daily_csv_path="DataAPI/data/QQQ_day.csv",
    k5m_csv_path="DataAPI/data/QQQ_5M.csv",
    code="QQQ",
    daily_chan_start="2010-02-11",
    accumulation_start="2012-02-11",
    N_confirm=5,
    min_labeled_days_to_train=200,
    retrain_every_new_labels=25,
    dp_lookback=5,
    lookahead_days_5m=2.0,
    retrain_every_days_5m=5,
    min_samples_total_5m=300,
    threshold_window_days=2.0,
    threshold_ret_grid=None,
    threshold_min_open_signals=10,
    initial_capital=100000.0,
    fee_pct=0.0,
    daily_chan_max_klines=500,
    five_chan_max_klines=500,
    macro_files={
        "vix_": "VIX.csv",
    },
    static_buy_level=0.20,
    static_sell_level=0.30,
    daily_threshold_config=RollingThresholdConfig(
        lookback_days=252,
        buy_grid=make_threshold_grid(0.05, 0.35, 0.005),
        sell_grid=make_threshold_grid(0.15, 0.60, 0.005),
        min_gap=0.02,
        min_obs=60,
        switch_penalty=0.0,
    ),
    verbose=True,
)

snapshot_res = build_adaptive_reward_snapshot(
    snapshot_path=str(snapshot_path),
    snapshot_end_time="2019-12-31",
    output_dir="output_adaptive_reward_snapshot_build_QQQ_252days_at_2020",
    **common_kwargs,
)

print(snapshot_res["snapshot_path"])
display(snapshot_res["daily_reward_df"].tail())
display(snapshot_res["daily_log_df"].tail())

resume_res = run_adaptive_reward_from_snapshot(
    dp_lookback_override=252,
    snapshot_path=str(snapshot_path),
    end_time="2026-12-31",
    sim_start="2020-01-01",
    initial_capital=100000.0,
    fee_pct=0.0,
    output_dir="output_adaptive_reward_resumed_fresh_QQQ_252days_at_2020",
    # Optional. If omitted, the code saves next to the source snapshot as
    # checkpoints/QQQ_adaptive_reward_fresh_start__continued.joblib
    #save_snapshot_path="output_adaptive_reward_resumed_fresh_QQQ_252days_at_2020/final_checkpoint.joblib",
    autosave_year_start_checkpoints=True,
    verbose=True,
)

print("continued snapshot:", resume_res["continued_snapshot_path"])

display(resume_res["daily_reward_df"].tail())
display(resume_res["daily_log_df"].tail())
display(resume_res["trades_df"].head())



[TRAIN][DAILY-PROB] n=200 pos=42 (21.00%)
[TRAIN][DAILY-PROB] n=225 pos=50 (22.22%)
[TRAIN][DAILY-PROB] n=250 pos=55 (22.00%)
[TRAIN][DAILY-PROB] n=275 pos=59 (21.45%)
[TRAIN][DAILY-PROB] n=300 pos=69 (23.00%)
[TRAIN][DAILY-PROB] n=325 pos=69 (21.23%)
[TRAIN][DAILY-PROB] n=350 pos=69 (19.71%)
[TRAIN][DAILY-PROB] n=375 pos=72 (19.20%)


KeyboardInterrupt: 